# Can we predict `passed_quiz`?
Plan (follows checklist.md):
- Rows have no timestamp and each student appears in several rows, so the test set holds **whole students** (grouped split), and CV inside training is grouped by student too.
- Identifier `student_key` is dropped. Features: `enrol_year`, `school_code`, `prior_score`, `session_minutes`, `hints_used`. `session_minutes` and `hints_used` are recorded during the session, before the closing quiz, so they are available when the quiz outcome is predicted (assumption: prediction is made at the quiz start). `prior_score` is described as stored on the session record; it varies within a student, so it is treated as a per-session value known at session time.
- Candidates named up front: (A) majority-class baseline, (B) logistic regression, (C) histogram gradient boosting. Small fixed hyperparameter grids.
- Selection metric: ROC AUC in CV; decision threshold chosen on out-of-fold predictions only.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score, f1_score
from sklearn.base import clone
df = pd.read_csv('data.csv')
TARGET, GROUP = 'passed_quiz', 'student_key'
FEATURES = ['enrol_year','school_code','prior_score','session_minutes','hints_used']
print(df.shape, df[TARGET].mean(), df[GROUP].nunique())

## 1. Hold out a test set of whole students (before any exploration or modelling)

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
tr_idx, te_idx = next(gss.split(df, groups=df[GROUP]))
train, test = df.iloc[tr_idx].reset_index(drop=True), df.iloc[te_idx].reset_index(drop=True)
assert set(train[GROUP]).isdisjoint(test[GROUP])
print(len(train), len(test), train[TARGET].mean())
X_train, y_train, g_train = train[FEATURES], train[TARGET], train[GROUP]

## 2-4. Candidates (named before scoring) and grouped CV on training rows only
Preprocessing (scaling, one-hot of `school_code`) lives inside the Pipeline, so it is refit in each fold.

In [ ]:
num = ['enrol_year','prior_score','session_minutes','hints_used']
def prep(): return ColumnTransformer([('num',StandardScaler(),num),('cat',OneHotEncoder(handle_unknown='ignore',min_frequency=20),['school_code'])])
candidates = {'baseline_majority': Pipeline([('m',DummyClassifier(strategy='prior'))])}
for C_ in [0.01,0.1,1.0]:
    candidates[f'logreg_C{C_}'] = Pipeline([('p',prep()),('m',LogisticRegression(C=C_,max_iter=1000))])
for lr,d in [(0.05,3),(0.05,None),(0.1,3)]:
    candidates[f'hgb_lr{lr}_depth{d}'] = Pipeline([('m',HistGradientBoostingClassifier(learning_rate=lr,max_depth=d,max_iter=200,early_stopping=True,random_state=0))])
cv = GroupKFold(n_splits=5)
folds = list(cv.split(X_train, y_train, groups=g_train))
oof, rows = {}, []
for name, est in candidates.items():
    p = np.zeros(len(train)); aucs=[]
    for a,b in folds:
        m = clone(est).fit(X_train.iloc[a], y_train.iloc[a])
        p[b] = m.predict_proba(X_train.iloc[b])[:,1]
        aucs.append(roc_auc_score(y_train.iloc[b], p[b]))
    oof[name]=p; rows.append((name,np.mean(aucs),np.std(aucs)))
cvres = pd.DataFrame(rows,columns=['model','cv_auc_mean','cv_auc_sd']).sort_values('cv_auc_mean',ascending=False)
cvres

## 5. Choose model and threshold on CV only

In [ ]:
best = cvres.iloc[0].model
p = oof[best]
ths = np.linspace(0.05,0.95,91)
bal = [balanced_accuracy_score(y_train, p>=t) for t in ths]
thr = float(ths[int(np.argmax(bal))])
print(best, 'threshold', thr, 'CV balanced acc', max(bal))

**Decision (written before touching the test set):** the model is whichever candidate has the highest mean grouped-CV AUC (printed above), with the decision threshold that maximises balanced accuracy on its out-of-fold predictions. Features are the five listed above; no further tuning.

## 6. Refit on all training rows, score the test set once

In [ ]:
final = clone(candidates[best]).fit(X_train, y_train)
pt = final.predict_proba(test[FEATURES])[:,1]
yt = test[TARGET]
print(f'TEST ROC AUC {roc_auc_score(yt,pt):.3f} (constant baseline = 0.5 by construction)')
print(f'TEST balanced acc @thr {balanced_accuracy_score(yt,pt>=thr):.3f}  accuracy {accuracy_score(yt,pt>=thr):.3f}  F1 {f1_score(yt,pt>=thr):.3f}')
print('majority-class accuracy', max(yt.mean(),1-yt.mean()))
# uncertainty: bootstrap over test students
rng=np.random.default_rng(0); gs=test[GROUP].unique(); idx=test.groupby(GROUP).indices; bs=[]
for _ in range(500):
    s=np.concatenate([idx[g] for g in rng.choice(gs,len(gs))]); bs.append(roc_auc_score(yt.iloc[s],pt[s]))
print('AUC 95% CI (student bootstrap):', np.percentile(bs,[2.5,97.5]).round(3))

## Conclusion
See the printed test results above: the headline is the test AUC/balanced accuracy of the single refit model. Caveats: no time column, so generalisation is to new students from the same population, not to later periods; `prior_score` varies within a student despite being a placement score, so its provenance is worth confirming; `hints_used`/`session_minutes` are valid only if predictions are made at quiz time.